In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.utils.config import DATA_PROCESSED, TARGET_COL, TIME_COL
from src.utils.plot_helpers import set_style, save_fig, PALETTE
from src.models.lgbm_model import (
    train_lgbm_quantiles, compute_shap_values, lgbm_point_fn
)
from src.models.cross_validation import run_cv
from src.models.forecast_store import save_forecasts

set_style()

df = pd.read_parquet("../data/processed/features_DE.parquet")
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).reset_index(drop=True)

print(f"Rows : {len(df):,}")
print(f"Range: {df[TIME_COL].min()} → {df[TIME_COL].max()}")

In [ ]:
models, val_preds, imp_df = train_lgbm_quantiles(df, val_fraction=0.15)

print("\nVal set scores:")
from src.metrics.evaluation import smape, mase
clean = val_preds.dropna(subset=["actual","q50"])
print(f"  sMAPE (q50) : {smape(clean['actual'], clean['q50']):.2f}%")
print(f"  MASE  (q50) : {mase(clean['actual'], clean['q50']):.3f}")

print("\nFeature importance (q50 model):")
print(imp_df.to_string(index=False))

In [ ]:
series = df.set_index(TIME_COL)[TARGET_COL].dropna()

print("Running LightGBM CV (5 folds × 24h)...")
cv_lgbm = run_cv(
    series, lgbm_point_fn,
    n_splits=5, horizon=24,
    min_train_size=24*7*8,
    model_name="LightGBM",
)
save_forecasts(cv_lgbm, "cv_lgbm")
print(f"Saved. Shape: {cv_lgbm.shape}")

In [ ]:
# Point forecast file (same format as classical models)
tft_point = val_preds[[TIME_COL, "actual", "q50"]].copy()
tft_point = tft_point.rename(columns={"q50": "forecast"})
tft_point["model"]     = "LightGBM"
tft_point["fold"]      = 0
tft_point["horizon_h"] = range(1, len(tft_point) + 1)
save_forecasts(
    tft_point[["fold","timestamp","actual","forecast","model","horizon_h"]],
    "cv_lgbm_point"
)

# Full quantile file (for Phase 7 calibration)
save_forecasts(val_preds, "lgbm_quantile_forecasts")
print("Saved quantile forecasts.")

In [ ]:
plot_df = val_preds.dropna(subset=["actual"]).head(24*7)
plot_df[TIME_COL] = pd.to_datetime(plot_df[TIME_COL])

fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(
    plot_df[TIME_COL], plot_df["q10"], plot_df["q90"],
    alpha=0.25, color=PALETTE["primary"],
    label="80% interval (q10–q90)",
)
ax.plot(plot_df[TIME_COL], plot_df["q50"],
        color=PALETTE["primary"], linewidth=1.5, label="Median (q50)")
ax.plot(plot_df[TIME_COL], plot_df["actual"],
        color="black", linewidth=1.2, alpha=0.8, label="Actual")
ax.set_title("LightGBM quantile forecasts — first 7 days of validation")
ax.set_ylabel("Load (MW)")
ax.legend()
plt.tight_layout()
save_fig(fig, "14_lgbm_quantile_forecast")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df["feature"], imp_df["importance_pct"],
        color=PALETTE["primary"], alpha=0.85)
ax.set_xlabel("Importance (%)")
ax.set_title("LightGBM feature importance (q50 model)")
ax.invert_yaxis()
plt.tight_layout()
save_fig(fig, "15_lgbm_feature_importance")
plt.show()

In [ ]:
df_feat = df.copy()
if "hour_sin" not in df_feat.columns:
    from src.models.lgbm_model import _add_time_features, _add_lag_features
    df_feat = _add_time_features(df_feat)
    df_feat = _add_lag_features(df_feat)

shap_vals, feat_names = compute_shap_values(
    models["q50"], df_feat, n_samples=500
)

fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_vals, df_feat[feat_names].dropna().head(500),
    feature_names=feat_names,
    show=False, plot_size=None,
)
plt.title("SHAP summary — LightGBM q50")
plt.tight_layout()
save_fig(fig, "16_shap_summary")
plt.show()

In [ ]:
import os
print("=" * 50)
print("PHASE 4 SUMMARY — LightGBM")
print("=" * 50)
clean = val_preds.dropna(subset=["actual","q50"])
from src.metrics.evaluation import smape, mase, coverage
print(f"sMAPE (q50)  : {smape(clean['actual'], clean['q50']):.2f}%")
print(f"MASE  (q50)  : {mase(clean['actual'], clean['q50']):.3f}")
print(f"Coverage 80% : {coverage(clean['actual'], clean['q10'], clean['q90'])*100:.1f}%")
print()
print("Files saved:")
for f in sorted(os.listdir("../outputs/forecasts")):
    print(f"  {f}")